# Homework 3
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

In [72]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.spatial import distance
from scipy.stats import wrapcauchy, levy_stable
import math


# Functions

In [73]:
# Nota: Esta clase la importaremos junto con el segundo bloque de modulos
################# http://www.pygame.org/wiki/2DVectorClass ##################
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)
     # Método para convertir el vector en una tupla
    def to_tuple(self):
        return (self.x, self.y)

In [74]:
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=5, s_pos=[0,0]):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
    Returns:
        BM_2d_df:
    """
    # Init velocity vector
    velocity =Vec2d(speed,0)
    
    # Init DF
    BM_2d_df = pd.DataFrame(columns=['x_pos','y_pos'])    
    # Add initial position
    temp_df = pd.DataFrame([{'x_pos':s_pos[0], 'y_pos':s_pos[1]}])    
    BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
    
    # Generate the trajectory
    for i in range(n_steps-1):        
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        velocity = velocity.rotated(turn_angle)
    
        temp_df = pd.DataFrame([{'x_pos':BM_2d_df.x_pos[i]+velocity.x, 'y_pos':BM_2d_df.y_pos[i]+velocity.y}])    
        BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
        
    return BM_2d_df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_2d(n_steps=1000, speed=5, s_pos=[0,0],c =0.5):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
    Returns:
        BM_2d_df:
    """
    # Init velocity vector
    velocity =Vec2d(speed,0)
    
    # Init DF
    BM_2d_df = pd.DataFrame(columns=['x_pos','y_pos'])    
    # Add initial position
    temp_df = pd.DataFrame([{'x_pos':s_pos[0], 'y_pos':s_pos[1]}])    
    BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
    
    # Generate the trajectory
    for i in range(n_steps-1):        
        turn_angle =   wrapcauchy.rvs(c)   
        velocity = velocity.rotated(turn_angle)
    
        temp_df = pd.DataFrame([{'x_pos':BM_2d_df.iloc[i]['x_pos'] + velocity.x, 'y_pos':BM_2d_df.y_pos[i]+velocity.y}])    
        BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
        
    return BM_2d_df

#####################################################################################
# Correlated Random Walk 
#####################################################################################
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c = 0.5):
    pos = Vec2d(0, 0)
    trajectory = [pos.to_tuple()]
    angle = 0  # Ángulo inicial en radianes
    for i in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))  # Tamaño del paso con Lévy
        delta_angle = wrapcauchy.rvs(c)  # Generar un ángulo con distribución de Cauchy
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    print("Primeros 5 puntos de la trayectoria:", trajectory[:5])  # Verifica si hay datos

    x, y = zip(*trajectory)
    z = np.linspace(0, 1, len(x))  # Crear un eje Z para la visualización 3D
    
    print("Cantidad de puntos generados:", len(x))  # Debe ser n_steps + 1

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', name='Lévy Flight'))
    fig.show()


# Activity 1: Path length - BM1 vs BM2 vs CRW

In [75]:
###############
# Path Length #
###############
def path_length(trajectory):
    """
    Calculate the total path length of a given trajectory.
    
    Parameters:
    trajectory: A pandas DataFrame containing the trajectory
    
    Returns:
    path_length: A numpy array containing the cumulative sum of the distances
    """
    # Get the Euclidean Distance
    distances = np.array([distance.euclidean(trajectory.iloc[i-1], trajectory.iloc[i]) for i in range(1, trajectory.shape[0])])
    # Get the Cumulative Sum of the stephs
    return np.cumsum(distances)



n_steps = 1000

# Definir las configuraciones de cada tipo de caminata
walks = {
    "BM_3": bm_2d(n_steps, speed=3),
    "BM_6": bm_2d(n_steps, speed=6),
    "CRW_5": rw_2d(n_steps, speed=5),
    "CRW_6": rw_2d(n_steps, speed=6),
    "Levy_1": levy_flight(n_steps, alpha=1),
    "Levy_07": levy_flight(n_steps, alpha=0.7)
}

# Calcular longitudes de los caminos
# path_lengths = {key: path_length(df) for key, df in walks.items()}
path_lengths = {key: path_length(df) for key, df in walks.items() if df is not None}

# Definir el ancho de línea para cada caso
line_widths = {"BM_3": 2, "BM_6": 8, "CRW_5": 2, "CRW_6": 2, "Levy_1": 2, "Levy_07": 2}

# Crear la figura
fig = go.Figure()

# Agregar trazas en un loop
for key, df in walks.items():
    if df is None:
        print(f"Advertencia: La caminata '{key}' es None y no se graficará.")
        continue  # Saltar esta iteración si df es None

    fig.add_trace(go.Scatter(
        x=df.index,
        y=path_lengths.get(key, []),  # Evitar error si path_lengths[key] no existe
        marker=dict(size=2),
        line=dict(width=line_widths[key]),
        mode='lines',
        name=f'Path length {key.replace("_", " ")}',
        showlegend=True
    ))

# Configuración del layout
fig.update_layout(
    title_text='Path length - (BM1 vs BM2 vs CRW)',
    autosize=False,
    width=900,
    height=500
)

# Mostrar la gráfica
fig.show()


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(1.5908048577001985), np.float64(-0.41693261078257376)), (np.float64(1.7606541095161439), np.float64(1.4416716237645573)), (np.float64(1.640053754422538), np.float64(1.4054930516102897)), (np.float64(0.5837968679749592), np.float64(0.9673670653390639))]
Cantidad de puntos generados: 1001


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(2.002974054247138), np.float64(-1.0237175382667376)), (np.float64(77.4068051525873), np.float64(223.0744954725001)), (np.float64(77.6380255830668), np.float64(222.74372663518588)), (np.float64(26.4142325166901), np.float64(46.18310476993901))]
Cantidad de puntos generados: 1001


Advertencia: La caminata 'Levy_1' es None y no se graficará.
Advertencia: La caminata 'Levy_07' es None y no se graficará.


# Activity 2: Lévy Distribution - N Different Curves

# Activity 3: Histograms + Curves


# Activity 4:  Step-length Distribution